# Outils d'analyse des données trafic

Code par Claude STEIN

L'dée est de se servir des données de stations de comptage pour determiner par regression les données permettant de calculer les différents paramétres de la section ou se situe le point de comptage. Ensuite en se basant sur les travaux et les calculs de Cyril Enault dans son rapport sur l'étallement de l'aire dijonnaise, nous pouvons en déduire les différents seuils des Niveau de Service (NSC).

## Import des Bibliothéques :

In [1]:
import polars as pl
import numpy as np

from datetime import datetime
from scipy.optimize import curve_fit

## Calcul des 4 NSC sur un diagramme fondamental callé sur des données SIREDO : 
Le principe de définition des seuils des 4 Niveaux de Service de Circulation (NSC) sur un diagramme fondamental dans le plan débit/vitesse \((Q, V)\)
Ces seuils permettent de découper la courbe en quatre zones distinctes de fonctionnement macroscopique du trafic.

1. Les Ratios Débit / Capacité (Q/C) retenus

Pour segmenter la branche fluide (la partie supérieure de la courbe du diagramme fondamental), l'ingénierie du trafic s'appuie sur deux seuils de débit conventionnels liés à la capacité \(C\)(le débit maximal de la voie) :
- 75 % de la capacité (\(0,75 \times C\) : Le seuil conventionnel de confort des concepteurs.
- 90 % de la capacité (\(0,90 \times C\) : Le seuil critique proche de l'instabilité.


2. Le mécanisme de projection sur le plan (Q, V)

Puisque le seul débit est ambigu (un même débit peut être observé en régime fluide ou congestionné), c'est la vitesse qui l'accompagne qui sert de critère de discrimination final :
- On identifie sur l'axe des abscisses (Débit \(Q\)) les points correspondant à \(0,75 * C\)\ et \(0,90 * C\) , ainsi que la capacité maximale \(C\).
- On projette verticalement ces valeurs sur la branche supérieure (fluide) de la courbe calibrée \(V = f(Q)\) du diagramme fondamental.
- On en déduit trois vitesses seuils clés : \(V~1~\) , \(V~2~\) et \(V~3~\) qui délimitent verticalement les 4 NSC.


3. Les 4 Niveaux de Service (NSC) résultants


| Niveau de Service | Zone sur le diagramme (Plan Débit/Vitesse) | Caractéristiques de l'écoulement |
| :--: | :--: | :--: |
| NSC 1 : Fluide |	Débit allant de \(0\) à \(0,75 * C\). Vitesse supérieure a \(V~1~\) ( proche de la Vitesse Libre) | Les conducteurs roulent à la  vitesse souhitée. Les interactions entre les véhicules sont minimes et la liberté de manoeuvre est totale. |
| NSC 2 : Dense |	Débit allant de \(0,75 * C\) à \(0,90 * C\). Vitesse comprise entre \(V~1~\) et \(V~2~\) | La liberté de manoeuvre diminue. La formation de pelotons d'automobilistes commence a restreindre le confort de la conduite |
| NSC 3 : Instable |	Débit supérieur à \(0,90 * C\) jusqu'à la capacité \(C\). Vitesse chutant entre \(V~2~\) et \(V~3~\) (Vitesse Critique) | Le Trafic est à saturation. La moindre perturbation(coup de frein, changement de file) peut rompre l'équilibre et déclencher une onde de choc |
| NSC 4 : Congestionné |	Branche inférieure (basse) de la courbe. Vitesse inférieure à \(V~3~\) (Vitesse Critique), le débit s'effondre vers 0. | Le régime est dit "forcé" ou "congestif". Les véhicules subissent des cycles d'arrêt et de redémarrage (accordéon), la densité est maximale. |


## Choix et Importation des données

In [7]:
# Import des données de trafic dans jupyter notebook
try:
    from tkinter import Tk
    from tkFileDialog import askopenfilenames
except:
    from tkinter import Tk
    from tkinter import filedialog

Tk().withdraw() # we don't want a full GUI, so keep the root window from appearing
files = filedialog.askopenfilenames() # show an "Open" dialog box and return the path to the selected file
#print(files)
for fn in files:
# Lecture des données de la station
    data = pl.scan_csv(fn, separator=",",ignore_errors=True, try_parse_dates=True)

In [8]:
data=data.collect()

## Nettoyage et agrégation par flag

In [9]:
def process_siredo_traffic(df: pl.DataFrame) -> pl.DataFrame:
    """
    Traite un DataFrame de données individuelles SIREDO :
    1. Ajout de flags de qualité (présence vs vitesse valide).
    2. Agrégation par station et par pas de temps (6 minutes).
    3. Calcul propre des débits et des vitesses moyennes.
    """
    
    # Étape 1 : Application des flags de qualification
    df_flagged = df.with_columns([
        # Flag 1 : Présence physique valide (exclut les erreurs fatales de capteurs : 
        # vitesses aberrantes hors limites physiques absolues ou longueurs négatives/nulles)
        (
            (pl.col("vitesse") >= 0) & 
            (pl.col("vitesse") <= 200) & 
            (pl.col("longueur") > 0)
        ).alias("is_valid_presence"),
        
        # Flag 2 : Vitesse exploitable pour les analyses statistiques 
        # (exclut les excès extrêmes ou les artefacts de couplage de boucles)
        (
            (pl.col("vitesse") >= 10) & 
            (pl.col("vitesse") <= 160)
        ).alias("is_valid_speed")
    ])
    
    # Étape 2 : Troncature temporelle et agrégation
    df_aggregated = (
        df_flagged
        .with_columns(
            # Découpage temporel standard SIREDO (ex: pas de 6 minutes)
            intervalle = pl.col("timestamp").dt.truncate("6m")
        )
        .group_by(["id_station", "intervalle"])
        .agg([
            # Débit brut (tous les enregistrements reçus)
            pl.len().alias("debit_brut"),
            
            # Débit réel/valide : on compte tous les véhicules ayant physiquement franchi 
            # la station (on ne supprime pas le véhicule du débit sous prétexte qu'il roule vite)
            pl.col("is_valid_presence").sum().alias("debit_valide"),
            
            # Vitesse moyenne calculée uniquement sur les véhicules dont la vitesse est valide
            pl.when(pl.col("is_valid_speed"))
              .then(pl.col("vitesse"))
              .mean()
              .alias("vitesse_moyenne"),
              
            # Indicateur de qualité : taux de rejet des vitesses sur l'intervalle
            (
                1.0 - (pl.col("is_valid_speed").sum() / pl.col("is_valid_presence").sum().clip(lower_bound=1))
            ).alias("taux_rejet_vitesse")
        ])
        .sort(["id_station", "intervalle"])
    )
    
    return df_aggregated


# ==========================================
# Exemple d'utilisation avec des données simulées
# ==========================================
if __name__ == "__main__":
   
    df_brut = data
    
    print("--- Données Brutes ---")
    print(df_brut)
    
    df_resultat = process_siredo_traffic(df_brut)
    
    print("\n--- Données Agrégées et Nettoyées (Pas de 6 min) ---")
    print(df_resultat)

--- Données Brutes ---
shape: (13_363_799, 5)
┌──────────┬─────────────────────┬─────┬─────┬─────┐
│ PME_ID   ┆ HORODATE_PUB        ┆ Q   ┆ T   ┆ V   │
│ ---      ┆ ---                 ┆ --- ┆ --- ┆ --- │
│ str      ┆ datetime[μs]        ┆ i64 ┆ i64 ┆ i64 │
╞══════════╪═════════════════════╪═════╪═════╪═════╡
│ A31 METZ ┆ 2020-02-02 23:18:20 ┆ 6   ┆ 2   ┆ 87  │
│ A31 METZ ┆ 2020-02-02 23:21:21 ┆ 6   ┆ 2   ┆ 86  │
│ A31 METZ ┆ 2020-02-02 23:24:20 ┆ 5   ┆ 2   ┆ 85  │
│ A31 METZ ┆ 2020-02-02 23:27:21 ┆ 8   ┆ 2   ┆ 90  │
│ A31 METZ ┆ 2020-02-02 23:30:20 ┆ 5   ┆ 2   ┆ 88  │
│ …        ┆ …                   ┆ …   ┆ …   ┆ …   │
│ MZO57.X2 ┆ 2021-01-18 23:03:21 ┆ 358 ┆ 12  ┆ 96  │
│ MZO57.X2 ┆ 2021-01-18 23:06:20 ┆ 358 ┆ 12  ┆ 96  │
│ MZO57.X2 ┆ 2021-01-18 23:09:20 ┆ 358 ┆ 12  ┆ 96  │
│ MZO57.X2 ┆ 2021-01-18 23:12:20 ┆ 358 ┆ 12  ┆ 96  │
│ MZO57.X2 ┆ 2021-01-18 23:15:20 ┆ 358 ┆ 12  ┆ 96  │
└──────────┴─────────────────────┴─────┴─────┴─────┘


ColumnNotFoundError: unable to find column "vitesse"; valid columns: ["PME_ID", "HORODATE_PUB", "Q", "T", "V"]

## Calage des données Siredo pour tracer le diagramme fondamental de la station
Utilisation

Il suffit d'appeler la fonction principale en spécifiant le modèle souhaité ("exp_pur", "exp_mixte", "pow_pur" ou "pow_mixte") :

### Exemple de calage avec prise en compte des Poids Lourds (Modèle Exponentiel Mixte)
```python
resultats_exp_mixte = caler_modele_siredo(df_agrégé, type_modele="exp_mixte")
print(resultats_exp_mixte)
```

### Exemple de calage avec le modèle de Puissance pur
```python
resultats_pow_pur = caler_modele_siredo(df_agrégé, type_modele="pow_pur")
print(resultats_pow_pur)
```

In [9]:
# =====================================================================
# 1. DÉFINITION DES MODÈLES MATHÉMATIQUES (5 MODÈLES)
# =====================================================================

def model_greenshields(k, vf, Kj):
    """Modèle linéaire classique de Greenshields : V = vf * (1 - K / Kj)"""
    return vf * np.maximum(0.0, 1.0 - k / Kj)

def model_exp_pur(k, a, b, alpha):
    """1. Modèle Exponentiel Généralisé Pur"""
    return a * np.exp(-b * (np.maximum(0.0, k) ** alpha))

def model_exp_mixte(X, a, b, alpha, Cvp):
    """2. Modèle Exponentiel Généralisé Mixte (avec proportion p de PL)"""
    k, p = X
    num = a * np.exp(-b * (np.maximum(0.0, k) ** alpha))
    den = 1.0 + (Cvp - 1.0) * p
    return num / den

def model_pow_pur(k, a, b, alpha):
    """3. Modèle Puissance Généralisé Pur"""
    return a + b * (np.maximum(0.0, k) ** alpha)

def model_pow_mixte(X, a, b, alpha, Cvp):
    """4. Modèle Puissance Généralisé Mixte (avec proportion p de PL)"""
    k, p = X
    num = a + b * (np.maximum(0.0, k) ** alpha)
    den = 1.0 + (Cvp - 1.0) * p
    return num / den


# =====================================================================
# 2. PIPELINE DE PRÉPARATION ET CALAGE UNIFIÉ
# =====================================================================

def caler_modele_siredo(df_agg: pl.DataFrame, type_modele: str = "exp_mixte") -> dict:
    """
    Prépare les données SIREDO et calibre le modèle choisi parmi :
    - 'greenshields' : V = vf * (1 - K / Kj)
    - 'exp_pur'      : V = a * exp(-b * K^alpha)
    - 'exp_mixte'    : V = (a * exp(-b * K^alpha)) / (1 + (Cvp - 1) * p)
    - 'pow_pur'      : V = a + b * K^alpha
    - 'pow_mixte'    : V = (a + b * K^alpha) / (1 + (Cvp - 1) * p)
    """
    # Validation du type de modèle
    modeles_valides = ["greenshields", "exp_pur", "exp_mixte", "pow_pur", "pow_mixte"]
    if type_modele not in modeles_valides:
        raise ValueError(f"Modèle inconnu. Choisissez parmi : {modeles_valides}")

    # 1. Extraction en DataFrame Pandas et calculs préalables
    df_pd = df_agg.drop_nulls(subset=["debit_vl", "debit_pl", "vitesse_moyenne"]).to_pandas()
    
    # Conversion en débits horaires (Q_6min * 10)
    df_pd["debit_vl_h"] = df_pd["debit_vl"] * 10.0
    df_pd["debit_pl_h"] = df_pd["debit_pl"] * 10.0
    df_pd["debit_total_h"] = df_pd["debit_vl_h"] + df_pd["debit_pl_h"]
    
    # Proportion de PL (p) et Densité totale K (veh/km)
    df_pd["p"] = df_pd["debit_pl_h"] / df_pd["debit_total_h"].clip(lower_bound=1.0)
    df_pd["k"] = df_pd["debit_total_h"] / df_pd["vitesse_moyenne"]
    
    # Filtrage des aberrations et conditions limites
    mask = (df_pd["vitesse_moyenne"] > 10.0) & (df_pd["debit_total_h"] > 50.0)
    df_clean = df_pd[mask]
    
    k_data = df_clean["k"].values
    p_data = df_clean["p"].values
    v_data = df_clean["vitesse_moyenne"].values
    
    # 2. Configuration des paramètres selon le modèle
    if type_modele == "greenshields":
        func = model_greenshields
        xdata = k_data
        p0 = [130.0, 120.0]
        bounds = ([90.0, 60.0], [150.0, 300.0])
    elif type_modele == "exp_pur":
        func = model_exp_pur
        xdata = k_data
        p0 = [130.0, 0.01, 1.0]
        bounds = ([90.0, 1e-6, 0.1], [150.0, 1.0, 3.0])
    elif type_modele == "exp_mixte":
        func = model_exp_mixte
        xdata = (k_data, p_data)
        p0 = [130.0, 0.01, 1.0, 2.5]
        bounds = ([90.0, 1e-6, 0.1, 1.0], [150.0, 1.0, 3.0, 6.0])
    elif type_modele == "pow_pur":
        func = model_pow_pur
        xdata = k_data
        p0 = [130.0, -0.5, 1.0]
        bounds = ([90.0, -100.0, 0.1], [150.0, 0.0, 3.0])
    elif type_modele == "pow_mixte":
        func = model_pow_mixte
        xdata = (k_data, p_data)
        p0 = [130.0, -0.5, 1.0, 2.5]
        bounds = ([90.0, -100.0, 0.1, 1.0], [150.0, 0.0, 3.0, 6.0])

    # 3. Exécution de l'optimisation (curve_fit)
    try:
        popt, pcov = curve_fit(func, xdata, v_data, p0=p0, bounds=bounds)
        
        # Prédiction et calcul du R2
        v_pred = func(xdata, *popt)
        ss_res = np.sum((v_data - v_pred) ** 2)
        ss_tot = np.sum((v_data - np.mean(v_data)) ** 2)
        r2 = 1.0 - (ss_res / ss_tot)
        
    except Exception as e:
        print(f"Erreur lors de l'optimisation pour le modèle '{type_modele}' : {e}")
        return {}

    # Restitution structurée des résultats
    resultat = {
        "modele": type_modele,
        "r2": r2,
        "covariance": pcov
    }
    
    if type_modele == "greenshields":
        resultat["params"] = {"vf": popt[0], "Kj": popt[1]}
    elif "mixte" in type_modele:
        resultat["params"] = {"a": popt[0], "b": popt[1], "alpha": popt[2], "Cvp": popt[3]}
    else:
        resultat["params"] = {"a": popt[0], "b": popt[1], "alpha": popt[2]}
        
    return resultat

## Fonction séparée de comparaison (AIC, BIC, $R^2$ ajusté)
Elle s'appuie sur des critères statistiques robustes pour départager les 4 formulations :

- Le $R^2$  ajusté : Contrairement au $R^2$ classique qui augmente artificiellement dès qu'on ajoute des variables ou des paramètres, le $R^2$  ajusté pénalise les modèles qui ont un nombre de paramètres plus élevé (par exemple, les modèles mixtes à 4 paramètres $a, b, \alpha, C_{vp}$ par rapport aux modèles purs à 3 paramètres).

- L'AIC (Akaike Information Criterion) et le BIC (Bayesian Information Criterion) : Ces critères évaluent la qualité de l'ajustement tout en pénalisant la complexité du modèle (le nombre de paramètres $k$ ). Le modèle optimal est celui qui présente les valeurs d'AIC et de BIC les plus faibles.

In [10]:
def comparer_modeles_siredo(resultats_dict: dict) -> pl.DataFrame:
    """
    Compare un dictionnaire de résultats de calage (provenant de caler_modele_siredo)
    à l'aide du R2 ajusté, de l'AIC et du BIC.
    
    Format attendu pour chaque entrée du dictionnaire :
    {
        "nom_modele": {
            "r2": float,
            "n_obs": int,
            "rss": float,        Residual Sum of Squares
            "k_params": int      Nombre de paramètres libres
        }
    }
    """
    comparaisons = []
    
    for nom, res in resultats_dict.items():
        if not res or "r2" not in res:
            continue
            
        n = res["n_obs"]
        k = res["k_params"]
        r2 = res["r2"]
        rss = res["rss"]
        
        # 1. R2 ajusté
        # R2_adj = 1 - [ (1 - R2) * (n - 1) / (n - k - 1) ]
        if n - k - 1 > 0:
            r2_ajustee = 1.0 - ((1.0 - r2) * (n - 1) / (n - k - 1))
        else:
            r2_ajustee = np.nan
            
        # 2. AIC (Akaike Information Criterion) avec correction petits échantillons (AICc)
        # AIC = n * ln(RSS / n) + 2k + [2k(k+1) / (n - k - 1)]
        if rss > 0 and n - k - 1 > 0:
            term_base = n * np.log(rss / n) + 2 * k
            correction = (2 * k * (k + 1)) / (n - k - 1)
            aicc = term_base + correction
            
            # BIC = n * ln(RSS / n) + k * ln(n)
            bic = n * np.log(rss / n) + k * np.log(n)
        else:
            aicc = np.nan
            bic = np.nan
            
        comparaisons.append({
            "Modèle": nom,
            "R²": round(r2, 4),
            "R² Ajusté": round(r2_ajustee, 4),
            "AICc": round(aicc, 2),
            "BIC": round(bic, 2),
            "Paramètres (k)": k
        })
        
    # Conversion en DataFrame Polars pour un affichage propre, trié par AICc croissant (le plus bas est le meilleur)
    df_comparaison = pl.DataFrame(comparaisons).sort("AICc")
    return df_comparaison